<a href="https://colab.research.google.com/github/kny1209/test2/blob/main/AI/DR-08079_%EA%B5%AC%EB%82%98%EC%98%81_AI%EC%9D%91%EC%9A%A9_13%EC%B0%A8%EC%8B%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## [문제 1] 주관식 단답형 (10점)

의료 영상 진단, 특히 폐렴이나 암과 같은 중증 질환의 1차 스크리닝 AI 모델을 개발할 때, 단순한 '정확도(Accuracy)'보다 '재현율(Recall, 또는 민감도 Sensitivity)'이 더 중요한 평가지표로 여겨지는 이유는 무엇입니까? 강의안의 내용을 바탕으로 서술하시오.

**[답안 작성란]:** 환자를 놓치지 않는 것이 매우 중요하기 때문이다. 치료를 받아야 하는 상황에 오진을 하면 심각한 결과를 초래하기 때문이다



## [문제 2] 주관식 단답형 (10점)

딥러닝 모델은 흔히 내부 동작을 알 수 없는 '블랙박스'로 불립니다. 의료 현장에서 AI의 판단 근거를 시각적으로 보여주어 의료진의 신뢰를 얻기 위해 사용하는 **XAI(설명 가능한 AI) 기법**으로, 강의에서 소개된 '모델이 이미지의 어느 부분을 중요하게 보았는지 히트맵으로 시각화하는 기법'의 명칭은 무엇입니까?

**[답안 작성란]:** Grad-CAM



## [문제 3] 코드 작성: 이미지 전처리 파이프라인 (20점)

ResNet18 모델에 입력하기 위해 흉부 X-ray 이미지를 전처리하는 `transforms`를 정의하시오.
**요구사항:**
1. 모든 이미지를 **224x224** 크기로 변경할 것.
2. 이미지를 PyTorch **Tensor**로 변환할 것.
3. ImageNet 데이터셋의 평균과 표준편차를 사용하여 **정규화(Normalize)** 할 것.
   - 평균: `[0.485, 0.456, 0.406]`
   - 표준편차: `[0.229, 0.224, 0.225]`

In [ ]:
import torchvision.transforms as transforms

transform = transforms.Compose([
    # 1. 이미지 크기 조정 (224x224)
    transforms.Resize((224,224)),

    # 2. 텐서 변환
    transforms.ToTensor(),

    # 3. 정규화 (Normalize)
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

## [문제 4] 코드 작성: ResNet18 전이학습 모델 설정 (20점)

사전 학습된(Pretrained) ResNet18 모델을 불러오고, 마지막 분류 계층(Fully Connected Layer)을 우리의 문제(NORMAL, PNEUMONIA 2개 클래스)에 맞게 수정하는 코드를 작성하시오.

In [ ]:
import torch.nn as nn
from torchvision import models
from torchvision.models import ResNet18_Weights

# 1. 사전 학습된 ResNet18 모델 로드
# TODO: models.resnet18을 사용하여 weights를 ResNet18_Weights.IMAGENET1K_V1로 설정하여 로드하세요.
model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

# 2. 마지막 FC Layer 수정 (입력 특징 개수는 유지하고, 출력 개수를 2로 변경)
num_ftrs = model.fc.in_features
# TODO: model.fc를 nn.Linear를 사용하여 2개의 출력을 가지도록 수정하세요.
model.fc = nn.Linear(num_ftrs, 2)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 140MB/s]


## [문제 5] 코드 작성: 학습 루틴 구현 (20점)

모델 학습(Training) 단계의 핵심 코드를 작성하시오. 주어진 `inputs`와 `labels`를 이용하여 **순전파(Forward), 손실 계산, 역전파(Backward), 가중치 갱신** 과정을 구현해야 합니다.

In [ ]:
import torch.optim as optim

# 손실 함수와 옵티마이저 정의 (가정)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 학습 루프의 일부라고 가정 (inputs, labels는 데이터로더에서 받았다고 가정)
# inputs, labels = ...

def train_step(model, inputs, labels, criterion, optimizer):
    # 1. 기울기(Gradient) 초기화
    optimizer.zero_grad()

    # 2. 순전파 (Forward Pass)
    # TODO: 모델에 inputs를 넣어 outputs를 구하세요.
    outputs = model(inputs)

    # 3. 손실(Loss) 계산
    # TODO: criterion을 사용하여 outputs와 labels 사이의 오차를 구하세요.
    loss = criterion(outputs, labels)

    # 4. 역전파 (Backward Pass) 및 가중치 갱신
    loss.backward()
    optimizer.step()

    return loss.item()

## [문제 6] 코드 작성: 재현율(Recall) 계산 (20점)

혼동 행렬(Confusion Matrix)의 결과가 다음과 같이 변수로 주어졌을 때, **재현율(Recall, TPR)**을 계산하는 코드를 작성하시오.

* `TP` (True Positive): 실제 폐렴을 폐렴으로 예측
* `FN` (False Negative): 실제 폐렴을 정상으로 오진
* `FP` (False Positive): 실제 정상을 폐렴으로 오진
* `TN` (True Negative): 실제 정상을 정상으로 예측

In [ ]:
# 예시 값 (실제 계산식은 변수를 사용해야 함)
TP = 388
FN = 2
FP = 120
TN = 114

# TODO: 재현율(Recall)을 계산하는 수식을 작성하세요.
recall = TP / (TP + FN)

print(f"Recall: {recall:.4f}")

Recall: 0.9949
